# ERA5 Ice Hazard Data Pipeline — Central Asia (2015-2024)

This notebook downloads, processes and merges ERA5 reanalysis data for ice hazard assessment across five Central Asian countries.

---

## What is ERA5?

ERA5 is the fifth generation global atmospheric reanalysis produced by the **European Centre for Medium-Range Weather Forecasts (ECMWF)** under the **Copernicus Climate Change Service (C3S)**.

A reanalysis combines numerical weather prediction models with historical observations (weather stations, satellites, radiosondes) to produce a consistent and continuous reconstruction of past climate conditions.

| Property | Value |
|---|---|
| Spatial resolution | ~31 km (0.25 x 0.25 degrees) |
| Temporal resolution | Hourly |
| Temporal coverage | 1940 to present |
| Vertical levels | 137 atmospheric levels |
| Source | ECMWF / Copernicus C3S |

---

## Variables

### 2m_temperature

- **Short name**: t2m
- **Unit**: Kelvin (K). Convert to Celsius: T(C) = T(K) - 273.15
- **Step type**: Instant
- **Description**: Air temperature at 2 metres above the surface. Standard surface temperature variable used in climatological studies.
- **Relevance for ice hazard**: Identifies freeze days (T < 0 C), cold spells, and freeze-thaw cycles that cause mechanical stress and ice accumulation on power lines. Using 4 time steps per day (00:00, 06:00, 12:00, 18:00) allows detection of intra-day freeze-thaw cycles, which are the primary driver of ice load fatigue on conductors.

### total_precipitation

- **Short name**: tp
- **Unit**: Metres (m) per time step. Convert to mm: P(mm) = P(m) x 1000
- **Step type**: Accumulated (reset at 00:00 UTC)
- **Description**: Total accumulated precipitation (rain and snow) over the time step.
- **Relevance for ice hazard**: Combined with temperature, identifies freezing rain events (tp > 0 and T < 0 C), which are the most critical scenario for power lines as they cause direct ice accretion on conductors and towers.

---

## Time Sampling Strategy

This pipeline uses **4 time steps per day**: 00:00, 06:00, 12:00, 18:00 UTC.

This choice is a balance between data volume and analytical accuracy:
- A single daily snapshot (e.g. 00:00 only) misses intra-day freeze-thaw cycles
- Hourly data (24 steps/day) produces very large files with marginal gain for regional-scale studies
- 4 steps/day captures the main thermal transitions (night minimum, morning warming, afternoon peak, evening cooling) and is standard practice for infrastructure climate risk assessments

---

## Study Area — Central Asia

| Country | Code | Bounding Box (N, W, S, E) |
|---|---|---|
| Tajikistan | TJK | 41.1 N, 67.3 E, 36.5 N, 75.2 E |
| Turkmenistan | TKM | 42.8 N, 52.2 E, 35.0 N, 66.8 E |
| Kyrgyzstan | KGZ | 43.3 N, 69.0 E, 39.0 N, 80.0 E |
| Kazakhstan | KAZ | 55.5 N, 46.0 E, 40.0 N, 87.5 E |
| Uzbekistan | UZB | 46.0 N, 55.0 E, 37.0 N, 74.0 E |

---

## Requirements

1. A registered account on the [Copernicus CDS](https://cds.climate.copernicus.eu/)
2. A configured `~/.cdsapirc` file containing your API credentials:
```
url: https://cds.climate.copernicus.eu/api
key: YOUR-API-KEY
```
3. Required packages: `cdsapi`, `xarray`, `h5netcdf`

---

## Pipeline Overview

1. Clean existing data directory
2. Download ERA5 data per country per year via the CDS API
3. Unzip the returned archive and merge the two variable files (instant + accumulated)
4. Merge all yearly files into a single file per country
5. Inspect and validate the output datasets

---
## 0. Imports and Parameters

In [1]:
import cdsapi
import os
import glob
import shutil
import zipfile
import xarray as xr

DATA_DIR = "data/era5/ice"
YEARS    = [str(y) for y in range(2015, 2025)]
TIMES    = ["00:00", "06:00", "12:00", "18:00"]

COUNTRIES = {
    "TJK": {"area": [41.1, 67.3, 36.5, 75.2]},
    "TKM": {"area": [42.8, 52.2, 35.0, 66.8]},
    "KGZ": {"area": [43.3, 69.0, 39.0, 80.0]},
    "KAZ": {"area": [55.5, 46.0, 40.0, 87.5]},
    "UZB": {"area": [46.0, 55.0, 37.0, 74.0]},
}

print("Parameters loaded")
print(f"  Countries  : {list(COUNTRIES.keys())}")
print(f"  Years      : {YEARS[0]} to {YEARS[-1]}")
print(f"  Time steps : {TIMES}")

Parameters loaded
  Countries  : ['TJK', 'TKM', 'KGZ', 'KAZ', 'UZB']
  Years      : 2015 to 2024
  Time steps : ['00:00', '06:00', '12:00', '18:00']


---
## 1. Clean Existing Data Directory

Remove all previously downloaded files (single time step 00:00 only) and start fresh.

In [2]:
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
    print(f"Removed: {DATA_DIR}")

os.makedirs(DATA_DIR, exist_ok=True)
print(f"Created: {os.path.abspath(DATA_DIR)}")

Removed: data/era5/ice
Created: /Users/nassimdekkar/Documents/RI-Infra-exposure/notebooks/data/era5/ice


---
## 2. Download ERA5 Data via CDS API

The CDS API returns a ZIP archive containing two separate NetCDF files:
- `data_stream-oper_stepType-instant.nc` — temperature (2m_temperature)
- `data_stream-oper_stepType-accum.nc` — precipitation (total_precipitation)

The script downloads, unzips, and merges these two files into a single NetCDF file per country per year. Already downloaded files are skipped.

In [3]:
c = cdsapi.Client()

for country, params in COUNTRIES.items():
    for year in YEARS:
        final_file = os.path.join(DATA_DIR, f"era5_ice_{country}_{year}.nc")

        if os.path.exists(final_file):
            print(f"Skip: {country} {year} — file already exists")
            continue

        print(f"Downloading: {country} {year}")
        tmp_zip = os.path.join(DATA_DIR, f"tmp_{country}_{year}.zip")
        tmp_dir = os.path.join(DATA_DIR, f"tmp_{country}_{year}")

        try:
            c.retrieve("reanalysis-era5-single-levels", {
                "product_type": "reanalysis",
                "variable": ["2m_temperature", "total_precipitation"],
                "year": year,
                "month": [f"{m:02d}" for m in range(1, 13)],
                "day":   [f"{d:02d}" for d in range(1, 32)],
                "time":  TIMES,
                "area":  params["area"],
                "format": "netcdf",
            }, tmp_zip)

            os.makedirs(tmp_dir, exist_ok=True)
            with zipfile.ZipFile(tmp_zip, "r") as z:
                z.extractall(tmp_dir)

            extracted = glob.glob(f"{tmp_dir}/*.nc")

            if len(extracted) == 1:
                shutil.move(extracted[0], final_file)
            else:
                # Merge temperature and precipitation into one file
                datasets = [xr.open_dataset(f) for f in extracted]
                xr.merge(datasets).to_netcdf(final_file)
                for ds in datasets:
                    ds.close()

            print(f"  Saved: {final_file}")

        finally:
            if os.path.exists(tmp_zip): os.remove(tmp_zip)
            if os.path.exists(tmp_dir): shutil.rmtree(tmp_dir)

print("Download complete")

Downloading: TJK 2015


2026-03-07 10:25:23,289 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-07 10:25:23,291 INFO Request ID is 5327310b-45a2-440d-b4e8-ae5de785e27d
2026-03-07 10:25:23,394 INFO status has been updated to accepted
2026-03-07 10:25:37,969 INFO status has been updated to running
2026-03-07 10:29:43,490 INFO status has been updated to successful


1efeda4fac50bc483c423c599ab0d4e7.zip:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/opt/anaconda3/lib/python3.12/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'rasterio' loading failed:
dlopen(/opt/anaconda3/lib/python3.12/site-packages/rasterio/_base.cpython-312-darwin.so, 0x0002): Library not loaded: @rpath/libnetcdf.19.dylib
  Referenced from: <A2860E87-DCA6-3E1C-869C-97EF1B16E899> /opt/anaconda3/lib/libgdal.32.3.6.2.dylib
  Reason: tried: '/opt/anaconda3/lib/libnetcdf.19.dylib' (no such file), '/opt/anaconda3/lib/python3.12/site-packages/rasterio/../../../libnetcdf.19.dylib' (no such file), '/opt/anaconda3/lib/python3.12/site-packages/rasterio/../../../libnetcdf.19.dylib' (no such file), '/opt/anaconda3/bin/../lib/libnetcdf.19.dylib' (no such file), '/opt/anaconda3/bin/../lib/libnetcdf.19.dylib' (no such file), '/usr/local/lib/libnetcdf.19.dylib' (no such file), '/usr/lib/libnetcdf.19.dylib' (no such file, not in dyld cache)
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


  Saved: data/era5/ice/era5_ice_TJK_2015.nc
Downloading: TJK 2016


...

abd497e0e89d601625dae85802d2d339.zip:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

  Saved: data/era5/ice/era5_ice_UZB_2023.nc
Downloading: UZB 2024


2026-03-07 17:28:37,104 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-03-07 17:28:37,105 INFO Request ID is 38ec6640-beb1-4471-8215-b3994a2bd54f
2026-03-07 17:28:37,179 INFO status has been updated to accepted
2026-03-07 17:28:51,002 INFO status has been updated to running
2026-03-07 17:31:29,816 INFO status has been updated to successful


9d0a05db08b6ba6a530626d37472eb1e.zip:   0%|          | 0.00/10.6M [00:00<?, ?B/s]

  Saved: data/era5/ice/era5_ice_UZB_2024.nc
Download complete


---
## 3. Verify Downloaded Files

In [4]:
files = sorted(glob.glob(f"{DATA_DIR}/era5_ice_*_20*.nc"))
print(f"{len(files)} files found:\n")
for f in files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {os.path.basename(f):<40} {size_mb:.1f} MB")

50 files found:

  era5_ice_KAZ_2015.nc                     33.1 MB
  era5_ice_KAZ_2016.nc                     33.6 MB
  era5_ice_KAZ_2017.nc                     32.6 MB
  era5_ice_KAZ_2018.nc                     32.5 MB
  era5_ice_KAZ_2019.nc                     32.6 MB
  era5_ice_KAZ_2020.nc                     32.4 MB
  era5_ice_KAZ_2021.nc                     32.1 MB
  era5_ice_KAZ_2022.nc                     32.7 MB
  era5_ice_KAZ_2023.nc                     32.4 MB
  era5_ice_KAZ_2024.nc                     33.1 MB
  era5_ice_KGZ_2015.nc                     3.1 MB
  era5_ice_KGZ_2016.nc                     3.1 MB
  era5_ice_KGZ_2017.nc                     3.1 MB
  era5_ice_KGZ_2018.nc                     3.1 MB
  era5_ice_KGZ_2019.nc                     3.1 MB
  era5_ice_KGZ_2020.nc                     3.1 MB
  era5_ice_KGZ_2021.nc                     3.0 MB
  era5_ice_KGZ_2022.nc                     3.1 MB
  era5_ice_KGZ_2023.nc                     3.0 MB
  era5_ice_KGZ_2024.nc 

---
## 4. Merge Yearly Files by Country

Concatenate all 10 yearly files (2015-2024) for each country into a single NetCDF file.

In [5]:
for country in COUNTRIES:
    output_file = f"{DATA_DIR}/era5_ice_{country}.nc"

    if os.path.exists(output_file):
        print(f"Skip: {country} — merged file already exists")
        continue

    print(f"Merging: {country}")
    files = sorted(glob.glob(f"{DATA_DIR}/era5_ice_{country}_20*.nc"))

    if not files:
        print(f"  Warning: no files found for {country}")
        continue

    merged = xr.open_mfdataset(files, combine="by_coords")
    merged.to_netcdf(output_file)
    print(f"  Saved: {output_file}")

print("Merge complete")

Merging: TJK
  Saved: data/era5/ice/era5_ice_TJK.nc
Merging: TKM
  Saved: data/era5/ice/era5_ice_TKM.nc
Merging: KGZ
  Saved: data/era5/ice/era5_ice_KGZ.nc
Merging: KAZ
  Saved: data/era5/ice/era5_ice_KAZ.nc
Merging: UZB
  Saved: data/era5/ice/era5_ice_UZB.nc
Merge complete


---
## 5. Inspect Merged Datasets

In [7]:
for country in COUNTRIES:
    path = f"{DATA_DIR}/era5_ice_{country}.nc"
    if not os.path.exists(path):
        continue

    ds = xr.open_dataset(path)
    time_dim = "valid_time" if "valid_time" in ds.dims else "time"

    print(f"\n{country}")
    print(f"  Variables      : {list(ds.data_vars)}")
    print(f"  Dimensions     : {dict(ds.sizes)}")
    print(f"  Period         : {str(ds[time_dim].values[0])[:10]} to {str(ds[time_dim].values[-1])[:10]}")
    print(f"  Time steps     : {len(ds[time_dim])} ({len(ds[time_dim]) // 365 // 10} steps/day approx.)")

    if "t2m" in ds:
        t_celsius = ds["t2m"] - 273.15
        print(f"  Temp min       : {float(t_celsius.min()):.1f} C")
        print(f"  Temp max       : {float(t_celsius.max()):.1f} C")
        freeze_steps = int((t_celsius < 0).sum())
        total_steps  = int(t_celsius.size)
        print(f"  Freeze steps   : {freeze_steps} / {total_steps} ({100*freeze_steps/total_steps:.1f}%)")

    if "tp" in ds:
        tp_mm = ds["tp"] * 1000
        print(f"  Precip max     : {float(tp_mm.max()):.2f} mm")

    ds.close()


TJK
  Variables      : ['tp', 't2m']
  Dimensions     : {'valid_time': 14612, 'latitude': 19, 'longitude': 31}
  Period         : 2015-01-01 to 2024-12-31
  Time steps     : 14612 (4 steps/day approx.)
  Temp min       : -43.1 C
  Temp max       : 47.0 C
  Freeze steps   : 3459950 / 8606468 (40.2%)
  Precip max     : 9.87 mm

TKM
  Variables      : ['tp', 't2m']
  Dimensions     : {'valid_time': 14612, 'latitude': 32, 'longitude': 59}
  Period         : 2015-01-01 to 2024-12-31
  Time steps     : 14612 (4 steps/day approx.)
  Temp min       : -35.7 C
  Temp max       : 49.3 C
  Freeze steps   : 2177297 / 27587456 (7.9%)
  Precip max     : 23.25 mm

KGZ
  Variables      : ['tp', 't2m']
  Dimensions     : {'valid_time': 14612, 'latitude': 18, 'longitude': 45}
  Period         : 2015-01-01 to 2024-12-31
  Time steps     : 14612 (4 steps/day approx.)
  Temp min       : -44.7 C
  Temp max       : 45.9 C
  Freeze steps   : 4307957 / 11835720 (36.4%)
  Precip max     : 13.04 mm

KAZ
  Variab

---
## 6. Output Summary

In [8]:
print("Merged files by country:\n")
for country in COUNTRIES:
    path = f"{DATA_DIR}/era5_ice_{country}.nc"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  OK      era5_ice_{country}.nc — {size_mb:.1f} MB")
    else:
        print(f"  MISSING era5_ice_{country}.nc")

Merged files by country:

  OK      era5_ice_TJK.nc — 24.3 MB
  OK      era5_ice_TKM.nc — 56.0 MB
  OK      era5_ice_KGZ.nc — 31.8 MB
  OK      era5_ice_KAZ.nc — 367.0 MB
  OK      era5_ice_UZB.nc — 94.4 MB
